# Setup

In [1]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from google import genai as google_genai
import time

# Ingest

In [2]:
# carrega variáveis de ambiente
load_dotenv()

QDRANT_ENDPOINT = os.getenv("QDRANT_ENDPOINT")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
GEMINI_API_KEY_T1 = os.getenv("GEMINI_API_KEY_T1")
COLLECTION_NAME = "ifrs-canoas"
REINGEST = False

# configura cliente Qdrant e google genai
client = QdrantClient(url=QDRANT_ENDPOINT, api_key=QDRANT_API_KEY)
google_client = google_genai.Client(api_key=GEMINI_API_KEY_T1)

print("Conexões configuradas.")

Conexões configuradas.


In [3]:
# carrega chunks
with open("../data/chunks/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Chunks carregados: {len(chunks)}")

Chunks carregados: 5783


In [4]:
# cria collection se não existir
if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"Collection '{COLLECTION_NAME}' criada.")
else:
    print(f"Collection '{COLLECTION_NAME}' já existe.")

Collection 'ifrs-canoas' já existe.


In [5]:
# busca source_urls já inseridos no Qdrant
def get_existing_urls():
    existing = set()
    offset = None
    
    while True:
        result, offset = client.scroll(
            collection_name=COLLECTION_NAME,
            with_payload=["source_url"],
            limit=1000,
            offset=offset
        )
        for point in result:
            existing.add(point.payload["source_url"])
        if offset is None:
            break
    
    return existing

# ingestão de chunks no qdrant com embedding via Gemini
def ingest_chunks(chunks, batch_size=100, start_id=0):
    total = len(chunks)
    for i in range(0, total, batch_size):
        batch = chunks[i:i + batch_size]
        texts = [c["text"] for c in batch]
        
        # embeda todos de uma vez com retry
        for attempt in range(5):
            try:
                result = google_client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=texts
                )
                break
            except Exception as e:
                if "429" in str(e):
                    wait = 30 * (attempt + 1)
                    print(f"  Rate limit, aguardando {wait}s...")
                    time.sleep(wait)
                else:
                    raise e
        
        points = []
        for j, (chunk, embedding) in enumerate(zip(batch, result.embeddings)):
            points.append(PointStruct(
                id=start_id + i + j,
                vector=embedding.values,
                payload={
                    "text": chunk["text"],
                    "source_url": chunk["source_url"],
                    "title": chunk["title"],
                    "type": chunk["type"],
                    "published_at": chunk.get("published_at")
                }
            ))
        
        client.upsert(collection_name=COLLECTION_NAME, points=points)
        print(f"Inseridos {min(i + batch_size, total)}/{total} chunks")

In [6]:
# lógica de reingest: 
# Se REINGEST for True, deleta e recria a collection do zero. 
# Se False, busca URLs já existentes e só insere os novos chunks.
if REINGEST:
    client.delete_collection(COLLECTION_NAME)
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print("Collection recriada do zero.")
    start_id = 0
    new_chunks = chunks
else:
    existing_urls = get_existing_urls()
    new_chunks = [c for c in chunks if c["source_url"] not in existing_urls]
    start_id = client.count(collection_name=COLLECTION_NAME).count
    print(f"Chunks novos a inserir: {len(new_chunks)}")

ingest_chunks(new_chunks, start_id=start_id)
print("Ingestão concluída.")

Chunks novos a inserir: 15
Inseridos 15/15 chunks
Ingestão concluída.


In [7]:
from qdrant_client.models import Filter, FieldCondition, MatchAny, PayloadSchemaType

url_alvo = "https://drive.google.com/file/d/1pQIT_vhyxsq49OJ8N71owZUBTj21Hxzo/view"

client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="source_url",
    field_schema=PayloadSchemaType.KEYWORD
)

results = client.scroll(
    collection_name=COLLECTION_NAME,
    scroll_filter=Filter(
        must=[FieldCondition(
            key="source_url",
            match=MatchAny(any=[url_alvo])
        )]
    ),
    with_payload=True,
    limit=10
)

print(f"Chunks no Qdrant: {len(results[0])}")
for p in results[0]:
    print(f"\n{p.payload['text'][:]}")
    print("---")

Chunks no Qdrant: 1

Ano documento: 2026
Fabiana Fidelis leciona Port. Ins. no F02 no TADS 1º semestre.
Fabiana Fidelis leciona Port. Ins. no LAB E08 (INF) no TADS 1º semestre.
Gustavo N. leciona Arq. Comp. no LAB E07 (INF) no TADS 2º semestre.
Denise leciona Prog. Estrut. no LAB E07 (INF) no TADS 2º semestre.
Aline Oliveira leciona Metod. Pesquisa no F10 no TADS 2º semestre.
Carla Mendes leciona Mat. Comp. 2 no F11 no TADS 2º semestre.
Jair Azevedo leciona Ling. Obj. I no LAB E10 (INF) no TADS 3º semestre.
Rafael Pinto leciona Est. Dados no LAB E10 (INF) no TADS 3º semestre.
Clayton Farias leciona Eng. Soft. no LAB E06 (ELE) no TADS 3º semestre.
Jacqueline Akazaki leciona Ban. Dad. no LAB E08 (INF) no TADS 3º semestre.
Gustavo N. leciona Sist. Oper. no LAB E10 (INF) no TADS 3º semestre.
Clayton Farias leciona Test. Soft. no LAB D05 (INF) no TADS 4º semestre.
Clayton Farias leciona Eng. Soft. II no LAB D10 (INF) no TADS 4º semestre.
Jair Azevedo leciona Ling. Obj. II no LAB D05 (INF) n